# Initial Data Exploration

This notebook performs initial exploratory data analysis on NYC TLC trip data.

## Objectives
1. Load and inspect raw data
2. Understand data structure and quality
3. Identify patterns and anomalies
4. Visualize key trends
5. Generate initial insights about hypothesis

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## 1. Load Data

In [ ]:
# Define data paths
RAW_DATA_PATH = Path('../../data/raw')
PROCESSED_DATA_PATH = Path('../../data/processed')

# List available files
raw_files = list(RAW_DATA_PATH.glob('*.parquet'))
print(f"Found {len(raw_files)} raw data files")
print("\nSample files:")
for f in raw_files[:5]:
    print(f"  - {f.name}")

In [ ]:
# Load a sample file for initial exploration
# Choose a recent month for yellow taxi
sample_file = [f for f in raw_files if 'yellow' in f.name][0]
print(f"Loading: {sample_file.name}")

df = pd.read_parquet(sample_file)
print(f"\nShape: {df.shape}")
print(f"Memory usage: {df.memory_usage().sum() / 1024**2:.2f} MB")

## 2. Data Structure and Quality

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Data types and null values
print("Data Info:")
df.info()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = 100 * missing / len(df)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
}).sort_values('Percentage', ascending=False)

print("Missing Values:")
print(missing_df[missing_df['Percentage'] > 0])

## 3. Temporal Patterns

In [ ]:
# Extract temporal features
if 'tpep_pickup_datetime' in df.columns:
    df['pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
elif 'lpep_pickup_datetime' in df.columns:
    df['pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])

df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek
df['day'] = df['pickup_datetime'].dt.day

In [ ]:
# Trips by hour of day
fig, ax = plt.subplots(figsize=(12, 6))
df['hour'].value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Trip Distribution by Hour of Day', fontsize=16)
ax.set_xlabel('Hour')
ax.set_ylabel('Number of Trips')
plt.tight_layout()
plt.show()

In [ ]:
# Trips by day of week
fig, ax = plt.subplots(figsize=(10, 6))
df['day_of_week'].value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Trip Distribution by Day of Week', fontsize=16)
ax.set_xlabel('Day of Week (0=Monday, 6=Sunday)')
ax.set_ylabel('Number of Trips')
plt.tight_layout()
plt.show()

## 4. Trip Characteristics

In [ ]:
# Distribution of trip distance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(df['trip_distance'], bins=50, edgecolor='black')
axes[0].set_title('Distribution of Trip Distance')
axes[0].set_xlabel('Distance (miles)')
axes[0].set_ylabel('Frequency')

# Box plot
axes[1].boxplot(df['trip_distance'].dropna())
axes[1].set_title('Trip Distance Box Plot')
axes[1].set_ylabel('Distance (miles)')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of fare amount
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(df['fare_amount'], bins=50, edgecolor='black')
axes[0].set_title('Distribution of Fare Amount')
axes[0].set_xlabel('Fare ($)')
axes[0].set_ylabel('Frequency')

# Box plot
axes[1].boxplot(df['fare_amount'].dropna())
axes[1].set_title('Fare Amount Box Plot')
axes[1].set_ylabel('Fare ($)')

plt.tight_layout()
plt.show()

## 5. Geographic Patterns

In [ ]:
# Top pickup locations
if 'PULocationID' in df.columns:
    top_pickup = df['PULocationID'].value_counts().head(10)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    top_pickup.plot(kind='barh', ax=ax)
    ax.set_title('Top 10 Pickup Locations', fontsize=16)
    ax.set_xlabel('Number of Trips')
    ax.set_ylabel('Location ID')
    plt.tight_layout()
    plt.show()